In [ ]:
import pandas as pd
import numpy as np


def entropy(s: pd.Series):
    values, counts = np.unique(s, return_counts=True)
    probabilities = counts / len(s)
    
    # Build the formula string
    terms = []
    for count, prob in zip(counts, probabilities):
        term = f"({count}/{len(s)}*log2({count}/{len(s)}))"
        terms.append(term)

    # Join terms with '+' and add the negative sign
    formula_str = "-(" + " + ".join(terms) + ")"

    # Calculate the entropy
    entropy_value = -np.sum(probabilities * np.log2(probabilities))

    # Print the formula and result
    print(f"{formula_str} = {entropy_value:.4f}")

    return entropy_value

def conditional_entropy(df: pd.DataFrame, feature: str, target: str):
    """Calculate weighted entropy H(target | feature) with detailed printout."""
    total_count = len(df)
    weighted_entropy = 0

    for value, group in df.groupby(feature):
        print(f"\nH({target} | {feature} = {value}):")
        entropy_value = entropy(group[target])
        weight = len(group) / total_count
        weighted_entropy += weight * entropy_value
        print(f"Weighted entropy contribution: ({len(group)}/{total_count}) * {entropy_value:.4f} = {weight * entropy_value:.4f}")

    print(f"\nWeighted Entropy H({target}|{feature}) = {weighted_entropy:.4f}")
    return weighted_entropy

def equal_width_binning(series: pd.Series, bins: int = 4) -> pd.Series:
    """
    Perform equal-width binning on a pandas Series.

    Parameters:
    - series: pd.Series - The input data series.
    - bins: int - The number of bins to divide the data into.

    Returns:
    - pd.Series of integer bin labels (0 to bins-1).
    """
    clean_series = series.dropna()
    binned = pd.cut(clean_series, bins=bins, labels=False, include_lowest=True)

    result = pd.Series(index=series.index, dtype='int64')
    result[clean_series.index] = binned
    return result

def equal_frequency_binning(series: pd.Series, bins: int = 4) -> pd.Series:
    """
    Perform equal-frequency (quantile) binning on a pandas Series.

    Parameters:
    - series: pd.Series - The input data series.
    - bins: int - The number of bins (quantiles).

    Returns:
    - pd.Series of integer bin labels (0 to bins-1).
    """
    clean_series = series.dropna()
    
    try:
        binned = pd.qcut(clean_series, q=bins, labels=False, duplicates='drop')
    except ValueError as e:
        print("Equal-frequency binning failed:", e)
        binned = pd.Series(index=clean_series.index, dtype='int64')  # empty

    result = pd.Series(index=series.index, dtype='int64')
    result[clean_series.index] = binned
    return result

In [ ]:
DATA = r'data.csv'
TARGET = 'משפחת התרופה'


df = pd.read_csv(DATA)
df['גיל'] = equal_width_binning(df['גיל'], bins=4)
df['ערכי סוכר'] = equal_frequency_binning(df['ערכי סוכר'], bins=4)

df

In [ ]:
features = list(df.columns)
features.remove(TARGET)

In [ ]:
entropy(df[TARGET])

In [ ]:
c_entropy = []
for feature in features:
    c_entropy.append((feature, conditional_entropy(df, feature, TARGET)))

In [ ]:
for feature, ce in sorted(c_entropy, key=lambda x: x[1]):
    print(f"{feature}: {ce:.4f}")